# Comparing Asset Pricing Models

In this notebook, we will replicate the relevant parts of Table 3 (and eventually Table 4) from Fama and French (2012).

We will estimate three asset pricing models:

1. CAPM
2. Fama-French three-factor model
3. Carhart four-factor model

We will use portfolios formed using company size and book-to-market ratio.

We will study three cases:

1. Global Developed portfolios with Global Developed factors
2. Japanese portfolios with Global Developed factors
3. Japanese portfolios with Japanese factors

For each case, we will first use all 25 portfolios. We will then remove the five portfolios in the smallest size group and repeat the analysis using the remaining 20 portfolios.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

In [ ]:
DATA_DIR = Path("cleaned_data")

## Load the factor data

The three-factor files contain:

- `Mkt-RF`: Market return minus the risk-free return
- `SMB`: Return of small companies minus large companies
- `HML`: Return of value companies minus growth companies
- `RF`: Risk-free return

The momentum files contain:

- `WML`: Return of past winners minus past losers

In [ ]:
developed_factors = pd.read_csv(
    DATA_DIR / "developed_3_factors.csv",
    parse_dates=["date"]
)

In [ ]:
developed_momentum = pd.read_csv(
    DATA_DIR / "developed_momentum.csv",
    parse_dates=["date"]
)

In [ ]:
japan_factors = pd.read_csv(
    DATA_DIR / "japan_3_factors.csv",
    parse_dates=["date"]
)

In [ ]:
japan_momentum = pd.read_csv(
    DATA_DIR / "japan_momentum.csv",
    parse_dates=["date"]
)

## Load the portfolio data

The portfolio files contain value-weighted monthly returns for 25 portfolios.

The portfolios are formed using:

- Five company-size groups
- Five book-to-market groups

This gives 5 × 5 = 25 portfolios for each market.

In [ ]:
developed_portfolios = pd.read_csv(
    DATA_DIR / "developed_25_size_bm.csv",
    parse_dates=["date"]
)

In [ ]:
japan_portfolios = pd.read_csv(
    DATA_DIR / "japan_25_size_bm.csv",
    parse_dates=["date"]
)

## Add the momentum factor

The momentum factor is stored in a separate file.

We merge the momentum data with the three-factor data using `date`.

The `one_to_one` check confirms that each month appears only once in each dataset.

In [ ]:
developed_factors = developed_factors.merge(
    developed_momentum,
    on="date",
    validate="one_to_one"
)

In [ ]:
japan_factors = japan_factors.merge(
    japan_momentum,
    on="date",
    validate="one_to_one"
)

## Check the datasets

The sample runs from November 1990 to March 2011.

Each dataset should contain 245 monthly observations.

The factor datasets should contain six columns:

- `date`
- `Mkt-RF`
- `SMB`
- `HML`
- `RF`
- `WML`

The portfolio datasets should contain one date column and 25 portfolio return columns.

In [ ]:
print("Developed factors:", developed_factors.shape)
print("Japanese factors:", japan_factors.shape)
print("Developed portfolios:", developed_portfolios.shape)
print("Japanese portfolios:", japan_portfolios.shape)

## Select the portfolios

The 5x5 results use all 25 portfolios.

The 4x5 results remove the five portfolios in the smallest size group.

The smallest size group appears in the first five portfolio columns. Removing these columns leaves 20 portfolios.

In [ ]:
all_portfolios = developed_portfolios.columns.drop("date").tolist()

In [ ]:
without_microcaps = all_portfolios[5:]

In [ ]:
print("Number of 5x5 portfolios:", len(all_portfolios))
print("Number of 4x5 portfolios:", len(without_microcaps))

## Define the models

The CAPM uses only the market factor.

The three-factor model adds size and value.

The four-factor model also adds momentum.

In [ ]:
models = {
    "CAPM": ["Mkt-RF"],
    "Three-factor": ["Mkt-RF", "SMB", "HML"],
    "Four-factor": ["Mkt-RF", "SMB", "HML", "WML"]
}

## Portfolio regressions

A separate regression is estimated for every portfolio.

The dependent variable is the portfolio's excess return:

Portfolio excess return = Portfolio return minus RF

The explanatory variables depend on the model.

### CAPM

The portfolio excess return is explained using `Mkt-RF`.

### Three-factor model

The portfolio excess return is explained using `Mkt-RF`, `SMB` and `HML`.

### Four-factor model

The portfolio excess return is explained using `Mkt-RF`, `SMB`, `HML` and `WML`.

Each regression includes a constant. This constant is the portfolio's alpha.

Alpha is the average monthly return that remains unexplained after accounting for the factors included in the model.

## Market and factor combinations

We will estimate the models for three combinations.

### Global portfolios with Global factors

Global Developed portfolio returns are explained using Global Developed factors.

### Japanese portfolios with Global factors

Japanese portfolio returns are explained using Global Developed factors.

### Japanese portfolios with Japanese factors

Japanese portfolio returns are explained using Japanese factors.

The second and third cases allow us to compare whether Japanese portfolio returns are explained better by Global or Japanese factors.

## Statistics reported in Table 3

Table 3 combines the results from the individual portfolio regressions.

The statistics reported depend on whether the analysis uses 25 or 20 portfolios.

### 5x5 results

The 5x5 results use all 25 portfolios.

The paper reports:

- `GRS`
- `|a|`
- `Adjusted R2`
- `s(a)`
- `SR(a)`

### 4x5 results

The 4x5 results remove the five portfolios in the smallest size group and use the remaining 20 portfolios.

The paper reports only:

- `GRS`
- `|a|`
- `SR(a)`

The paper does not report `Adjusted R2` or `s(a)` for the 4x5 cases.

### GRS statistic

The GRS statistic tests whether all portfolio alphas are jointly equal to zero.

A larger GRS value gives more evidence that the model leaves unexplained returns.

This statistic is not provided directly by the individual regression outputs. We calculate it using:

- The alphas from all portfolio regressions
- The residuals from all portfolio regressions
- The factor returns
- The number of portfolios
- The number of factors
- The number of observations

For the 5x5 case, calculate one GRS statistic using all 25 regressions together.

For the 4x5 case, calculate one GRS statistic using all 20 regressions together.

Do not calculate a separate GRS statistic for each portfolio and then average it.

### Average absolute alpha: |a|

Alpha is the constant reported by each portfolio regression. It measures the average monthly return that the model does not explain.

The alpha can be taken directly from the fitted regression output.

To calculate `|a|`:

1. Collect the alpha from every regression.
2. Take the absolute value of every alpha.
3. Calculate the simple average.

For the 5x5 case, average the 25 absolute alphas.

For the 4x5 case, average the 20 absolute alphas.

A smaller value means that the model leaves less unexplained return.

### Average adjusted R-squared

Adjusted R-squared measures how much of the monthly movement in a portfolio's excess return is explained by the model.

It can be taken directly from each fitted regression output.

For the 5x5 results:

1. Collect the adjusted R-squared from each of the 25 regressions.
2. Calculate the simple average.

A higher average adjusted R-squared means that the model explains more of the variation in portfolio returns.

The paper reports this statistic only for the 5x5 results. It is not reported for the 4x5 results.

### Average standard error of alpha: s(a)

The standard error of alpha shows how precisely the alpha has been estimated.

It can be taken directly from each fitted regression output as the standard error of the constant.

For the 5x5 results:

1. Collect the standard error of alpha from each of the 25 regressions.
2. Calculate the simple average.

A smaller value means that the portfolio alphas are estimated more precisely.

The paper reports this statistic only for the 5x5 results. It is not reported for the 4x5 results.

### Sharpe ratio of the alphas: SR(a)

`SR(a)` measures the combined size of the portfolio alphas relative to the risk in the regression residuals.

To calculate it:

1. Collect the alpha from every portfolio regression.
2. Place the alphas in one vector.
3. Collect the residuals from every regression.
4. Use the residuals to calculate the residual covariance matrix.
5. Combine the alpha vector and residual covariance matrix using the formula provided in the paper.
6. Take the square root of the resulting value.

For the 5x5 case, use all 25 alphas and their residuals.

For the 4x5 case, use the 20 alphas and their residuals.

This produces one joint `SR(a)` value for each model. It is not the average of separate portfolio Sharpe ratios.

The exact formula for `SR(a)` is provided in the paper.

## Producing the Table 3 results

For each task:

1. Select the required portfolio and factor data.
2. Merge the datasets using `date`.
3. Subtract `RF` from every portfolio return.
4. Add a constant to the explanatory variables.
5. Run a separate regression for every selected portfolio.
6. Store the fitted regression results for each model.
7. Use the stored results to calculate the required Table 3 statistics.
8. Place the three models in the rows of one table.
9. Round the final values to two decimal places.

For the 5x5 tasks, report:

- `GRS`
- `|a|`
- `Adjusted R2`
- `s(a)`
- `SR(a)`

For the 4x5 tasks, report only:

- `GRS`
- `|a|`
- `SR(a)`

## Task 1: Global portfolios with Global factors, 5x5

Use:

- `developed_portfolios`
- `developed_factors`
- All 25 portfolios in `all_portfolios`

For each portfolio, run:

1. CAPM
2. Three-factor model
3. Four-factor model

Store the regression results and calculate the five Table 3 statistics.

Print one table containing the results for the three models.

Compare your results with the Global 5x5 part of Table 3. State whether your values match and interpret the main differences between the models.

In [ ]:
# Write your code here



## Task 2: Global portfolios with Global factors, 4x5

Use:

- `developed_portfolios`
- `developed_factors`
- The 20 portfolios in `without_microcaps`

For each portfolio, run the CAPM, three-factor model and four-factor model.

Store the regression results.

For each model, report the three statistics shown in the 4x5 part of Table 3:

- `GRS`
- `|a|`
- `SR(a)`

Print one table containing the results for the three models.

Compare your results with the Global 4x5 part of Table 3. Discuss whether removing the five smallest portfolios changes the results.

In [ ]:
# Write your code here



## Task 3: Japanese portfolios with Global factors, 5x5

Use:

- `japan_portfolios`
- `developed_factors`
- All 25 portfolios in `all_portfolios`

For each Japanese portfolio, run the CAPM, three-factor model and four-factor model using Global Developed factors.

Store the regression results and calculate the five Table 3 statistics.

Print one table containing the results for the three models.

Compare your results with the Japan, Global factors, 5x5 part of Table 3. Interpret how well Global factors explain Japanese portfolio returns.

In [ ]:
#Write your code here



## Task 4: Japanese portfolios with Global factors, 4x5

Use:

- `japan_portfolios`
- `developed_factors`
- The 20 portfolios in `without_microcaps`

For each Japanese portfolio, run the CAPM, three-factor model and four-factor model using Global Developed factors.

Store the regression results.

For each model, report the three statistics shown in the 4x5 part of Table 3:

- `GRS`
- `|a|`
- `SR(a)`

Print one table containing the results for the three models.

Compare your results with the Japan, Global factors, 4x5 part of Table 3. Discuss whether removing the smallest Japanese portfolios changes the results.

In [ ]:
# Write your code here



## Task 5: Japanese portfolios with Japanese factors, 5x5

Use:

- `japan_portfolios`
- `japan_factors`
- All 25 portfolios in `all_portfolios`

For each Japanese portfolio, run the CAPM, three-factor model and four-factor model using Japanese factors.

Store the regression results and calculate the five Table 3 statistics.

Print one table containing the results for the three models.

Compare your results with the Japan, Local factors, 5x5 part of Table 3. Compare these results with Task 3 and determine whether Japanese factors explain Japanese returns better than Global factors.

In [ ]:
# Write your code here



## Task 6: Japanese portfolios with Japanese factors, 4x5

Use:

- `japan_portfolios`
- `japan_factors`
- The 20 portfolios in `without_microcaps`

For each Japanese portfolio, run the CAPM, three-factor model and four-factor model using Japanese factors.

Store the regression results.

For each model, report the three statistics shown in the 4x5 part of Table 3:

- `GRS`
- `|a|`
- `SR(a)`

Print one table containing the results for the three models.

Compare your results with the Japan, Local factors, 4x5 part of Table 3.

Discuss:

1. Which model performs best.
2. Whether removing the smallest portfolios changes the results.
3. Whether Global or Japanese factors explain Japanese portfolio returns better.

In [ ]:
#Write your code here



# Table 4: Individual Portfolio Alphas

Table 3 summarizes the results across all 25 or 20 portfolio regressions.

Table 4 looks inside these summary results. It reports the alpha and alpha t-statistic for each individual portfolio.

Therefore:

- Table 3 tells us whether a model performs well overall.
- Table 4 shows which particular portfolios cause the model to perform well or poorly.

For example, Table 4 can show whether a model has difficulty explaining the returns of small growth, small value, big growth or big value portfolios.

## What Table 4 reports

For each portfolio, Table 4 reports:

- `a`: The regression alpha
- `t(a)`: The t-statistic of the alpha

Both values are taken directly from the fitted regression output.

The alpha is the regression constant.

The t-statistic is calculated as:

Alpha divided by the standard error of alpha.

Unlike Table 3, these values are not averaged across portfolios.

The 25 alphas and 25 t-statistics are arranged in 5 × 5 matrices.

The rows represent company size:

1. Small
2. Size group 2
3. Size group 3
4. Size group 4
5. Big

The columns represent book-to-market:

1. Low
2. Group 2
3. Group 3
4. Group 4
5. High

## Task 7: Global portfolio alphas using Global factors

You have already estimated and stored the regression outputs for the Global portfolios using Global factors in Task 1.

Use the stored 5x5 regression results for all 25 portfolios.

For each of the following models, report:

- The alpha of each portfolio
- The t-statistic of each alpha

Models:

1. CAPM
2. Three-factor model
3. Four-factor model

Arrange the 25 alphas in a 5 × 5 matrix.

Arrange the 25 alpha t-statistics in another 5 × 5 matrix.

Compare your results with the section titled:

`Global size-B/M returns regressed on global factors`

in Table 4.

In [ ]:
# Write your code here



## Task 8: Japanese portfolio alphas using Japanese factors

You have already estimated and stored the regression outputs for the Japanese portfolios using Japanese factors in Task 5.

Use the stored 5x5 regression results for all 25 portfolios.

Table 4 reports only the local Japanese three-factor model in this section.

Using the stored three-factor regression results, report:

- The alpha of each portfolio
- The t-statistic of each alpha

Arrange the 25 alphas in a 5 × 5 matrix.

Arrange the 25 alpha t-statistics in another 5 × 5 matrix.

Compare your results with the section titled:

`Japanese size/B-M returns regressed on Japanese factors`

in Table 4.

In [ ]:
# Write your code here

